# pRF
samples energy outputs from the steerable pyramid model using population receptive field (pRF) data for specific brain regions (e.g., V1, V2) and saves the results

**For Iking**

I run the code on my visual studio code lenovo laptop- have all files in a usb drive.
however all the necessery files are in our lab's google drive folder-

Under My Drive-> V1_Drift:
1. prfsample_expand- my outputs from running the prf sampling for subject 1 
2. NSD-> pyramind_expand - steerable pyramind outputs for images 0-15 (in the driver its 72k images so 4 TeraByte)

Rest of the stuff i think you know- each subjects files of betas and prf and stuff are in its own folder under NSD folder, and the expirement itself is in NSD->nsddata->expirements->nsd.

i run with 10 cores and 10 images in a batch and for some of the participant its for 10k images (those with 40 session).



In [ ]:
import os
import time
import numpy as np
import nibabel as nib  # Used to load .nii.gz neuroimaging files
import scipy.io as sio  # For loading MATLAB .mat files
import h5py  # For working with HDF5 files
import gc  # For manual garbage collection to free memory
from joblib import Parallel, delayed  # For parallel processing
from tqdm import tqdm  # For progress bars
import tempfile, shutil  # For safely saving output files
import psutil  # RAM/Disk guards & monitors


from google.colab import drive
drive.mount('/content/drive', force_remount=True)


# -----------------------------------------------------------------------
# Global toggles (for speed, keep these False)
# -----------------------------------------------------------------------
VERBOSE = False          # If True, show extra debug / sanity info
MONITOR_USAGE = False    # If True, print RAM/disk usage per batch

# -----------------------------------------------------------------------
# Paths config
# -----------------------------------------------------------------------
nsd_folder = "/content/drive/MyDrive/V1_Drift/NSD"  # Path to NSD dataset
pyramid_folder = "/content/drive/MyDrive/V1_Drift/NSD/pyramid_expand"  # Folder with steerable pyramid outputs
prf_folder = "/content/drive/MyDrive/V1_Drift/prfsample_expand"  # Folder to save output of pRF sampling

# -----------------------------------------------------------------------
# General settings
# -----------------------------------------------------------------------
interp_img_size = 714  # Interpolated image size used in model
background_size = 1024  # Size of the padded image canvas
img_scaling = 0.5  # Image scaling factor
num_levels = 7  # Number of pyramid spatial frequency levels
num_orientations = 8  # Number of orientation filters
pix_per_deg = img_scaling * 714 / 8.4  # pixels per degree
deg_per_pix = 8.4 / (714 * img_scaling)  # degrees per pixel

roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]
test_mode = False  # Toggle test mode (fewer images) for debugging

# Limit BLAS threads globally (avoid oversubscription with joblib)
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# -----------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------
def show_usage(tag=""):
    if not MONITOR_USAGE:
        return
    ram_gb = psutil.Process().memory_info().rss / 1024**3
    total, used, free = shutil.disk_usage("/content")
    print(
        f"[{tag}] RAM {ram_gb:5.2f} GB | "
        f"Disk used {used/1e9:6.1f}/{total/1e9:6.1f} GB | "
        f"Free {free/1e9:6.1f} GB"
    )

def enforce_cache_budget(cache_path="/content/pyr_cache",
                         max_bytes=3_000_000_000,
                         exts=(".npy", ".npz", ".pkl"),
                         verbose=False):
    """Trim disk cache under `max_bytes` by deleting oldest files recursively."""
    if not os.path.isdir(cache_path):
        return
    files = []
    for root, _, names in os.walk(cache_path):
        for n in names:
            if n.endswith(exts):
                fp = os.path.join(root, n)
                try:
                    files.append((fp, os.path.getmtime(fp), os.path.getsize(fp)))
                except FileNotFoundError:
                    pass
    total = sum(sz for _, _, sz in files)
    if total <= max_bytes:
        return
    files.sort(key=lambda t: t[1])  # oldest first
    if verbose:
        print(
            f"[CACHE] Over budget: {total/1e9:.2f} GB > "
            f"{max_bytes/1e9:.2f} GB. Trimming..."
        )
    for fp, _, sz in files:
        try:
            os.remove(fp)
            total -= sz
            if verbose:
                print(
                    f"  deleted {os.path.basename(fp)} "
                    f"({sz/1e6:.1f} MB) -> {total/1e9:.2f} GB"
                )
            if total <= max_bytes:
                break
        except FileNotFoundError:
            continue

def enforce_ram_budget(min_free_gb=2.0, on_pressure=None, tag=""):
    """If free RAM is low, run GC and optional callback that clears in-memory caches."""
    avail_gb = psutil.virtual_memory().available / 1024**3
    if avail_gb < min_free_gb:
        if VERBOSE:
            print(
                f"[RAM] Low free RAM ({avail_gb:.2f} GB). "
                f"Triggering GC{' + callback' if on_pressure else ''}... ({tag})"
            )
        gc.collect()
        if on_pressure:
            on_pressure()

def set_globals(roiGaus_shared):
    global GLOBAL_roiGaus
    GLOBAL_roiGaus = roiGaus_shared

def load_nifti(filepath):
    """Load a NIfTI file if it exists; return None otherwise."""
    return nib.load(filepath).get_fdata() if os.path.exists(filepath) else None

# ---- Worker thread limiting (avoid NumPy×joblib oversubscription) ----
_LIMIT_THREADS_DONE = False
def _limit_threads_once():
    """Ensure single-threaded BLAS inside each worker process."""
    global _LIMIT_THREADS_DONE
    if _LIMIT_THREADS_DONE:
        return
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
    _LIMIT_THREADS_DONE = True

# Define a local cache directory for unzipped .npz files
cache_dir = "/content/pyr_cache"
os.makedirs(cache_dir, exist_ok=True)

def load_pyramid_data(imgNum):
    """
    Load steerable pyramid data (sumOri, modelOri) from .npz,
    and cache as .npy files for faster subsequent access.

    Args:
        imgNum (int): Image number (1-based)

    Returns:
        tuple: (sumOri, modelOri) as np.float32
    """
    idx = imgNum - 1
    npz_path = os.path.join(pyramid_folder, f"pyrImg{idx}.npz")
    cached_sumOri = os.path.join(cache_dir, f"sumOri_{idx}.npy")
    cached_modelOri = os.path.join(cache_dir, f"modelOri_{idx}.npy")

    using_cache = os.path.exists(cached_sumOri) and os.path.exists(cached_modelOri)

    if using_cache:
        sumOri = np.load(cached_sumOri, allow_pickle=True)
        modelOri = np.load(cached_modelOri, allow_pickle=True)
    else:
        with np.load(npz_path, allow_pickle=True) as data:
            sumOri = data["sumOri"].astype(np.float32)
            modelOri = data["modelOri"]
            # Fix format if modelOri is wrapped in a 1-element array
            if (
                isinstance(modelOri, np.ndarray)
                and modelOri.shape == (1,)
                and isinstance(modelOri[0], list)
            ):
                modelOri = modelOri[0]
        np.save(cached_sumOri, sumOri)
        np.save(cached_modelOri, modelOri)

    return sumOri, modelOri

def process_single_image(imgNum, num_levels, num_orientations):
    _limit_threads_once()
    try:
        sumOri, modelOri = load_pyramid_data(imgNum)
    except Exception as e:
        if VERBOSE:
            print(f"[{imgNum}] load error: {e}")
        return imgNum, {}, {}

    prfSampleLev_partial = {}
    prfSampleLevOri_partial = {}

    L_sum = min(num_levels + 2, len(sumOri))
    L_ori = min(num_levels, len(modelOri))

    TARGET_L = num_levels + 2
    TARGET_O = num_orientations

    # Stack sums (L_sum, H, W)
    try:
        sum_stack = np.stack(sumOri[:L_sum], axis=0).astype(np.float32, copy=False)
        sum_stack = np.ascontiguousarray(sum_stack)
    except Exception as e:
        if VERBOSE:
            print(f"[{imgNum}] sum_stack error: {e}")
        return imgNum, {}, {}

    # Stack orientations (L_ori, O, H, W)
    try:
        ori_list = []
        for ilev in range(L_ori):
            om = modelOri[ilev]
            if isinstance(om, list):
                om = np.stack(om, axis=0)
            elif isinstance(om, np.ndarray) and om.ndim == 3:
                pass
            else:
                continue
            om = np.ascontiguousarray(om.astype(np.float32, copy=False))
            ori_list.append(om)
        if len(ori_list) > 0:
            max_O = max(arr.shape[0] for arr in ori_list)
            target_O_local = max(TARGET_O, max_O)
            fixed = []
            for arr in ori_list:
                if arr.shape[0] < target_O_local:
                    pad_O = target_O_local - arr.shape[0]
                    arr = np.pad(arr, ((0, pad_O), (0, 0), (0, 0)))
                fixed.append(arr)
            ori_stack = np.stack(fixed, axis=0).astype(np.float32, copy=False)
            ori_stack = np.ascontiguousarray(ori_stack)
        else:
            ori_stack = None
    except Exception as e:
        if VERBOSE:
            print(f"[{imgNum}] ori_stack error: {e}")
        return imgNum, {}, {}

    for roinum, G_all in GLOBAL_roiGaus.items():
        try:
            G_all = np.ascontiguousarray(G_all.astype(np.float32, copy=False))
            V, H, W = G_all.shape

            # ---------- Levels (keep +2 bands) ----------
            prf_levels = np.einsum("vhw,lhw->vl", G_all, sum_stack, optimize=True)
            # pad/trim to TARGET_L
            if prf_levels.shape[1] < TARGET_L:
                prf_levels = np.pad(
                    prf_levels, ((0, 0), (0, TARGET_L - prf_levels.shape[1]))
                )
            elif prf_levels.shape[1] > TARGET_L:
                prf_levels = prf_levels[:, :TARGET_L]
            prf_levels = prf_levels.astype(np.float32, copy=False)

            # ---------- Orientations (NO +2 here) ----------
            if ori_stack is not None and L_ori > 0:
                prf_oris = np.einsum("vhw,lohw->vlo", G_all, ori_stack, optimize=True)
                # trim/pad to exactly num_levels
                if prf_oris.shape[1] > num_levels:
                    prf_oris = prf_oris[:, :num_levels, :]
                elif prf_oris.shape[1] < num_levels:
                    prf_oris = np.pad(
                        prf_oris,
                        ((0, 0), (0, num_levels - prf_oris.shape[1]), (0, 0)),
                    )
                # pad orientations to num_orientations if needed
                if prf_oris.shape[2] < num_orientations:
                    prf_oris = np.pad(
                        prf_oris,
                        (
                            (0, 0),
                            (0, 0),
                            (0, num_orientations - prf_oris.shape[2]),
                        ),
                    )
            else:
                prf_oris = np.zeros(
                    (V, num_levels, num_orientations), dtype=np.float32
                )

            prf_oris = np.ascontiguousarray(prf_oris.astype(np.float32, copy=False))

            prfSampleLev_partial[roinum] = prf_levels
            prfSampleLevOri_partial[roinum] = prf_oris

        except Exception as e:
            if VERBOSE:
                print(f"[{imgNum}] ROI {roinum} einsum error: {e}")
            return imgNum, {}, {}

    try:
        del sum_stack
    except NameError:
        pass
    try:
        del ori_stack
    except NameError:
        pass
    gc.collect()

    return imgNum, prfSampleLev_partial, prfSampleLevOri_partial

def prf_sample_model_expand(isub, visualRegion, batch_size=100):
    """
    Main function to compute pRF samples by applying ROI Gaussians to steerable pyramid outputs.

    Args:
        isub: Subject index (1-based).
        visualRegion: Visual region group ID (1 to 4).
        batch_size: Number of images to process in parallel per batch.
    """
    print(f"\n-- Running pRF Sampling for Subject {isub}, Region {visualRegion} ---")

    # Output h5 file for saving processed pRF samples
    h5_filename = os.path.join(
        prf_folder, f"prfSampleStim_v{visualRegion}_sub{isub}.h5"
    )
    # Remove legacy .log file if exists
    log_path = os.path.join(prf_folder, f"processed_images_sub{isub}.log")
    if os.path.exists(log_path):
        os.remove(log_path)

    # Load NSD experimental design data
    nsdDesignPath = os.path.join(
        "/content/drive/MyDrive/V1_Drift/NSD/nsddata/experiments/nsd/nsd_expdesign.mat"
    )
    if not os.path.exists(nsdDesignPath):
        print(f"ERROR: NSD Design file not found: {nsdDesignPath}")
        return

    nsdDesign = sio.loadmat(nsdDesignPath)
    subjectim = nsdDesign["subjectim"]
    masterordering = nsdDesign["masterordering"].flatten() - 1
    valid_masterordering = masterordering[masterordering < subjectim.shape[1]]

    # trial-level (with repeats) and unique image IDs
    trial_imgids = subjectim[isub - 1, valid_masterordering].astype(np.int32)
    allImgs = np.unique(trial_imgids)

    if test_mode:
        allImgs = allImgs[:2000]

    print(f"Total unique images to consider: {len(allImgs)}")

    # Max number of processed images you expect to store in memmap log
    max_images = 10000
    log_mmap_path = os.path.join(prf_folder, f"processed_images_sub{isub}.mmap")

    # Initialize or load memory-mapped log file
    if os.path.exists(log_mmap_path):
        log_mmap = np.memmap(log_mmap_path, dtype="int32", mode="r+")
        current_log_index = np.count_nonzero(log_mmap)
        processed_images = set(int(x) for x in log_mmap[:current_log_index])
    else:
        log_mmap = np.memmap(log_mmap_path, dtype="int32", mode="w+", shape=(max_images,))
        log_mmap[:] = 0
        log_mmap.flush()
        current_log_index = 0
        processed_images = set()

    # ------------------------------------------------------------
    # Load pRF maps and ROI labels
    # ------------------------------------------------------------
    betas_folder = os.path.join(nsd_folder, f"subj{isub:02d}", "func1pt8mm")
    prf_files = [
        "prf_angle.nii.gz",
        "prf_eccentricity.nii.gz",
        "prf_size.nii.gz",
        "R2.nii.gz",
    ]

    angData, eccData, sizeData, r2Data = [
        load_nifti(os.path.join(betas_folder, f)) for f in prf_files
    ]
    visRoiData = load_nifti(os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz"))

    if any(data is None for data in [angData, eccData, sizeData, r2Data, visRoiData]):
        print("ERROR: Missing pRF data. Check file paths.")
        return

    # ------------------------------------------------------------
    # Define ROIs per visualRegion
    # ------------------------------------------------------------
    roi_map = {
        1: [0, 1],  # V1v + V1d
        2: [2, 3],  # V2v + V2d
        3: [4, 5],  # V3v + V3d
        4: [6],     # hV4
    }
    rois = roi_map.get(visualRegion, [])

    # ------------------------------------------------------------
    # Setup output containers and load cached Gaussians if available
    # ------------------------------------------------------------
    gauss_file = os.path.join(prf_folder, f"roiGaussians_sub{isub}_v{visualRegion}.h5")
    roiGaus = {}
    roiPrf = {}
    missing_rois = []

    if os.path.exists(gauss_file):
        if VERBOSE:
            print(f"Loading cached Gaussians from {gauss_file}")
        with h5py.File(gauss_file, "r") as f:
            for roinum in rois:
                key = f"roi_{roinum}"
                if key in f:
                    G_all = f[key][:]
                    roiGaus[roinum] = G_all

                    roi_mask = visRoiData == (roinum + 1)
                    ecc_deg = eccData[roi_mask]
                    ang_deg = angData[roi_mask]
                    sz_deg = sizeData[roi_mask]
                    sz_pix = sz_deg * pix_per_deg

                    theta = np.deg2rad(ang_deg)
                    ecc_pix = ecc_deg * pix_per_deg

                    x0_pix = ecc_pix * np.cos(theta)
                    y0_pix = ecc_pix * np.sin(theta)

                    roiPrf[roinum] = {
                        "ecc": ecc_deg,
                        "ang": ang_deg,
                        "sz": sz_deg,
                        "x": x0_pix,
                        "y": y0_pix,
                    }
                    voxel_ids = np.flatnonzero(roi_mask.ravel()).astype(np.int32)
                    roiPrf[roinum]["voxel_ids"] = voxel_ids
                else:
                    missing_rois.append(roinum)
    else:
        missing_rois = rois

    # ------------------------------------------------------------
    # Compute missing Gaussian kernels
    # ------------------------------------------------------------
    if missing_rois:
        if VERBOSE:
            print(f"Computing Gaussians for missing ROIs: {missing_rois}")

        half = background_size * img_scaling / 2.0
        coords = np.arange(-half + 0.5, half + 0.5, 1.0)
        x = coords
        y = coords
        X, Y = np.meshgrid(x, -y, indexing="xy")

        with h5py.File(gauss_file, "a") as f:
            for roinum in missing_rois:
                roi_mask = visRoiData == (roinum + 1)
                if np.sum(roi_mask) == 0:
                    if VERBOSE:
                        print(f"Skipping empty ROI: {roinum}")
                    continue

                ecc_deg = eccData[roi_mask]
                ang_deg = angData[roi_mask]
                sz_deg = sizeData[roi_mask]
                sz_pix = sz_deg * pix_per_deg

                theta = np.deg2rad(ang_deg)
                ecc_pix = ecc_deg * pix_per_deg

                x0_pix = ecc_pix * np.cos(theta)
                y0_pix = ecc_pix * np.sin(theta)

                roiPrf[roinum] = {
                    "ecc": ecc_deg,
                    "ang": ang_deg,
                    "sz": sz_deg,
                    "x": x0_pix,
                    "y": y0_pix,
                }

                X_diff = X.reshape(1, 512, 512) - x0_pix[:, None, None]
                Y_diff = Y.reshape(1, 512, 512) - y0_pix[:, None, None]

                G_all = np.exp(
                    -(X_diff**2 + Y_diff**2) / (2 * sz_pix[:, None, None] ** 2)
                )

                roiGaus[roinum] = G_all
                f.create_dataset(f"roi_{roinum}", data=G_all, compression="gzip")

        if VERBOSE:
            print("Updated Gaussian cache with missing ROIs")

    # ------------------------------------------------------------
    # Determine images to process (skip already processed; skip missing pyramid)
    # ------------------------------------------------------------
    work_list = []
    image_index_map = {}
    kept_imgs = []

    existing_all_imgs = None
    if os.path.exists(h5_filename):
        with h5py.File(h5_filename, "r") as f:
            if "allImgs" in f:
                existing_all_imgs = f["allImgs"][:]
                # Build a quick map: imgID -> index
                existing_idx = {
                    int(img): idx for idx, img in enumerate(existing_all_imgs)
                }
            else:
                existing_idx = None
    else:
        existing_idx = None

    for imgNum in allImgs:
        if imgNum in processed_images:
            continue

        pyramid_filename = os.path.join(pyramid_folder, f"pyrImg{imgNum - 1}.npz")
        if not os.path.exists(pyramid_filename):
            if VERBOSE:
                print(f"Missing pyramid file: {pyramid_filename}")
            continue

        work_list.append(imgNum)

        if existing_all_imgs is not None:
            # Use the index from the existing allImgs array
            idx = existing_idx.get(int(imgNum), None)
            if idx is None:
                raise ValueError(f"Image {imgNum} not found in existing allImgs.")
            image_index_map[imgNum] = idx
        else:
            # New file: index is just the next position in kept_imgs
            image_index_map[imgNum] = len(kept_imgs)

        kept_imgs.append(imgNum)

    if not work_list:
        print("\nNo new images to process. Skipping save.")
        return

    print(f"Processing {len(work_list)} new images")

    # ------------------------------------------------------------
    # Create / update HDF5 metadata + output datasets
    # ------------------------------------------------------------
    with h5py.File(h5_filename, "a") as f:
        if "allImgs" not in f:
            f.create_dataset("allImgs", data=np.array(kept_imgs, dtype=np.int32))

        if "trial_imgids" not in f:
            f.create_dataset(
                "trial_imgids",
                data=trial_imgids.astype(np.int32),
                compression="gzip",
            )

        if "rois" not in f:
            f.create_dataset("rois", data=np.array(rois, dtype=np.int32))

        f.attrs["numLevels"] = num_levels
        f.attrs["numOrientations"] = num_orientations
        f.attrs["interpImgSize"] = interp_img_size
        f.attrs["backgroundSize"] = background_size
        f.attrs["pixPerDeg"] = pix_per_deg
        f.attrs["roi_names"] = np.array(roi_names, dtype="S10")

        n_images = int(f["allImgs"].shape[0])

        # Create output datasets once
        for roinum, G_all in roiGaus.items():
            nvox = G_all.shape[0]
            dlev_key = f"prfSampleLev/roi_{roinum}"
            dori_key = f"prfSampleLevOri/roi_{roinum}"
            if dlev_key not in f:
                f.create_dataset(
                    dlev_key,
                    shape=(n_images, nvox, num_levels + 2),
                    maxshape=(n_images, nvox, num_levels + 2),
                    dtype=np.float32,
                    compression="gzip",
                )
            if dori_key not in f:
                f.create_dataset(
                    dori_key,
                    shape=(n_images, nvox, num_levels, num_orientations),
                    maxshape=(n_images, nvox, num_levels, num_orientations),
                    dtype=np.float32,
                    compression="gzip",
                )

        # voxel_ids per ROI
        if "voxel_ids" not in f:
            f.create_group("voxel_ids")
        for roinum in roiPrf:
            key_ids = f"voxel_ids/roi_{roinum}"
            if key_ids not in f:
                vid = roiPrf[roinum].get("voxel_ids")
                if vid is None:
                    roi_mask = visRoiData == (roinum + 1)
                    vid = np.flatnonzero(roi_mask.ravel()).astype(np.int32)
                f.create_dataset(
                    key_ids, data=vid, dtype=np.int32, compression="gzip"
                )

    # Pre-compute number of jobs once
    n_jobs = min(max(1, (os.cpu_count() or 2) - 1), 8)
    n_jobs = min(n_jobs, batch_size)

    num_batches = len(work_list) // batch_size + (
        1 if len(work_list) % batch_size != 0 else 0
    )

    start_time_global = time.time()
    MAX_RUNTIME_SECONDS = 23.5 * 3600  # Safety margin before 24h

    # ------------------------------------------------------------
    # Process images in batches
    # ------------------------------------------------------------
    for batch_num in range(num_batches):
        batch = work_list[batch_num * batch_size : (batch_num + 1) * batch_size]
        if not batch:
            continue
        if VERBOSE:
            print(
                f"Processing batch {batch_num + 1}/{num_batches} "
                f"with {len(batch)} images"
            )

        start = time.time()

        set_globals(roiGaus)

        results = Parallel(
            n_jobs=min(n_jobs, len(batch)),
            backend="loky",
            prefer="processes",
            batch_size="auto",
            verbose=0,
        )(
            delayed(process_single_image)(imgNum, num_levels, num_orientations)
            for imgNum in tqdm(
                batch,
                desc=f"Batch {batch_num + 1}/{num_batches}",
                unit="image",
                ncols=100,
                mininterval=0.5,
                disable=not VERBOSE,  # if you want *no* progress bar, set VERBOSE=False
            )
        )

        end = time.time()
        if VERBOSE:
            print(f"Batch {batch_num + 1} elapsed time: {end - start:.2f} s")

        # Write results directly to HDF5
        with h5py.File(h5_filename, "a") as f:
            for imgNum, partial_lev, partial_ori in results:
                if not partial_lev:
                    continue
                iimg = image_index_map[imgNum]
                for roinum in partial_lev:
                    f[f"prfSampleLev/roi_{roinum}"][iimg, :, :] = partial_lev[
                        roinum
                    ].astype(np.float32, copy=False)
                    f[f"prfSampleLevOri/roi_{roinum}"][iimg, :, :, :] = partial_ori[
                        roinum
                    ].astype(np.float32, copy=False)

        del results
        gc.collect()
        enforce_ram_budget(min_free_gb=2.0, tag=f"after batch {batch_num}")
        # Trim cache only once per batch (not per image)
        enforce_cache_budget(cache_dir, max_bytes=3_000_000_000, verbose=False)
        show_usage(f"batch {batch_num} saved")

        # Save pRF parameters once (idempotent)
        with h5py.File(h5_filename, "a") as f:
            if "prfParam" not in f:
                f.create_group("prfParam")

            for roinum in roiPrf:
                for param in ["ecc", "ang", "sz", "x", "y"]:
                    key = f"prfParam/{param}/roi_{roinum}"
                    data = roiPrf[roinum][param]
                    if key not in f:
                        f.create_dataset(key, data=data, compression="gzip")

            for roinum in roiPrf:
                key = f"prfParam/r2/roi_{roinum}"
                if key not in f:
                    roi_mask = visRoiData == (roinum + 1)
                    r2_roi = r2Data[roi_mask]
                    f.create_dataset(key, data=r2_roi, compression="gzip")

        # Update log for newly processed images
        new_logged_imgs = [img for img in batch if img not in processed_images]
        if new_logged_imgs:
            n = len(new_logged_imgs)
            if current_log_index + n > max_images:
                raise RuntimeError("Exceeded max_images capacity in memmap log.")
            log_mmap[current_log_index : current_log_index + n] = new_logged_imgs
            current_log_index += n
            log_mmap.flush()
            processed_images.update(new_logged_imgs)
            if VERBOSE:
                print(f"Logged {n} new images (mmap)")

        elapsed_global = time.time() - start_time_global
        if elapsed_global > MAX_RUNTIME_SECONDS:
            print(
                "Nearing Colab 24h timeout. Exiting early to save progress safely."
            )
            break

# ------------------------------------------------------------
# Script entry point
# ------------------------------------------------------------
if __name__ == "__main__":
    prf_sample_model_expand(isub=1, visualRegion=1, batch_size=20)


Check prfSampleStim_v1_sub1.h5 is valid

In [ ]:
import os, h5py, numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --------- config ----------
h5_path    = "/content/drive/MyDrive/V1_Drift/prfsample_expand/prfSampleStim_v1_sub3.h5"
roi_id     = 1
img_indices = [0, 1, 2]           # exactly 3 images (indices into allImgs)
vox_indices = [10, 200, 500]      # any set of voxels you want to overlay
SHOW_SUM_ORI_PER_LEVEL = False    # if True, dashed line shows Σ(orientations) per level
# ---------------------------

import os
import h5py
import numpy as np

def check_prf_hdf5_integrity(h5_path, verbose=True, check_values=True, atol=1e-7):
    """
    Validate structure, duplicates, corruption, and data consistency
    for prfSampleStim_vX_subY.h5 files.

    Args:
        h5_path (str): Path to the HDF5 file.
        verbose (bool): Print detailed results.
        check_values (bool): Check numeric sanity (mean/std, NaNs).
        atol (float): Tolerance for zero/NaN detection.
    """

    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"File not found: {h5_path}")

    issues = []

    try:
        f = h5py.File(h5_path, "r")
    except Exception as e:
        raise IOError(f"Cannot open file: {e}")

    if verbose:
        print(f"\n🔍 Checking file: {h5_path}")
        print("=" * 80)

    # --- 1. Basic structure ---
    required_groups = [
        "allImgs",
        "trial_imgids",
        "rois",
        "prfSampleLev",
        "prfSampleLevOri",
        "prfParam",
    ]

    for g in required_groups:
        if g not in f:
            issues.append(f"Missing group or dataset: {g}")
            if verbose:
                print(f" Missing: {g}")

    # --- 2. allImgs vs trial_imgids consistency ---
    if "allImgs" in f and "trial_imgids" in f:
        allImgs = f["allImgs"][:]
        trial_imgids = f["trial_imgids"][:]
        uniq_trials = np.unique(trial_imgids)
        if not np.array_equal(np.sort(allImgs), np.sort(uniq_trials)):
            issues.append("Mismatch: allImgs ≠ unique(trial_imgids)")
            if verbose:
                print(f" allImgs mismatch: {len(allImgs)} vs {len(uniq_trials)} unique trial IDs")
        else:
            if verbose:
                print(f" allImgs consistent ({len(allImgs)} images)")

    # --- 3. ROI dataset consistency ---
    if "rois" in f:
        rois = f["rois"][:]
        if verbose:
            print(f"ROIs listed: {rois.tolist()}")

        for roinum in rois:
            lev_key = f"prfSampleLev/roi_{roinum}"
            ori_key = f"prfSampleLevOri/roi_{roinum}"

            if lev_key not in f or ori_key not in f:
                issues.append(f"Missing dataset for ROI {roinum}")
                continue

            lev = f[lev_key]
            ori = f[ori_key]

            # check shape alignment
            if lev.shape[0] != ori.shape[0]:
                issues.append(f"Image count mismatch: {lev_key} vs {ori_key}")
            if lev.shape[1] != ori.shape[1]:
                issues.append(f"Voxel count mismatch: {lev_key} vs {ori_key}")

            if verbose:
                print(f"ROI {roinum}: prfLev {lev.shape}, prfLevOri {ori.shape}")

            # optional data checks
            if check_values:
                try:
                    sub = lev[:: max(1, len(lev)//10)]  # sample ~10%
                    mean_val = np.nanmean(sub)
                    std_val = np.nanstd(sub)
                    if np.isnan(mean_val) or np.isinf(mean_val):
                        issues.append(f"NaN/Inf values in {lev_key}")
                    if abs(mean_val) < atol and std_val < atol:
                        issues.append(f"All zeros or constant in {lev_key}")
                except Exception as e:
                    issues.append(f"Corrupt data in {lev_key}: {e}")

    # --- 4. prfParam consistency ---
    if "prfParam" in f:
        params = list(f["prfParam"].keys())
        if verbose:
            print(f"Parameter subgroups: {params}")

        # detect duplicate param keys (possible corruption)
        seen = set()
        for p in params:
            if p in seen:
                issues.append(f"Duplicate param subgroup: {p}")
            seen.add(p)

        # check per-ROI parameter length consistency
        for param in ["ecc", "ang", "sz", "x", "y", "r2"]:
            if param not in f["prfParam"]:
                issues.append(f"Missing parameter {param}")
                continue

            roi_grp = f["prfParam"][param]
            for roi_name, dset in roi_grp.items():
                vals = dset[:]
                if np.any(np.isnan(vals)):
                    issues.append(f"NaN in {param}/{roi_name}")
                if np.any(np.isinf(vals)):
                    issues.append(f"Inf in {param}/{roi_name}")
                if np.all(np.abs(vals) < atol):
                    issues.append(f"{param}/{roi_name} all zeros")

    # --- 5. Attribute sanity checks ---
    attrs = f.attrs
    required_attrs = ["numLevels", "numOrientations", "pixPerDeg", "roi_names"]
    for a in required_attrs:
        if a not in attrs:
            issues.append(f"Missing attribute: {a}")
        elif verbose:
            print(f"{a}: {attrs[a]}")

    # --- 6. Duplicate dataset detection (generic) ---
    all_keys = []
    f.visit(all_keys.append)
    if len(all_keys) != len(set(all_keys)):
        issues.append("Duplicate dataset paths found")

    # --- 7. Quick summary ---
    if verbose:
        print("\nSummary:")
        if issues:
            for i in issues:
                print(" ", i)
            print(f"Total issues found: {len(issues)}")
        else:
            print(" No problems detected — structure and data look consistent.")

    f.close()
    return issues


# --- Example usage ---
h5_path = "/content/drive/MyDrive/V1_Drift/prfsample_expand/prfSampleStim_v1_sub3.h5"
issues = check_prf_hdf5_integrity(h5_path)
if issues:
   print("\n File has potential issues — see list above.")
else:
   print("\nFile integrity confirmed clean.")



with h5py.File(h5_path, "r") as f:
    print("pixPerDeg:", f.attrs["pixPerDeg"])
